# MyDigitalTwin — TikTok
**Notebook 05 — Ingestion, exploration, nettoyage → Parquet**

Source : `data/raw/TIKTOK/user_data_tiktok.json`

Outputs :
- `data/parquet/tiktok_watch.parquet`
- `data/parquet/tiktok_likes.parquet`
- `data/parquet/tiktok_searches.parquet`
- `data/parquet/tiktok_comments.parquet`
- `data/parquet/tiktok_messages_meta.parquet`
- `data/parquet/tiktok_messages_text.parquet`

## Objectifs ML
- **Clone NLP (axe 1)** : commentaires + messages texte
- **ALS (axe 2)** : watch history + likes
- **K-Means (axe 3)** : activité temporelle

## 0. Initialisation

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone
import json, hashlib

spark = SparkSession.builder \
    .appName("MyDigitalTwin - TikTok") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

RAW_PATH    = "../../../data/raw/TIKTOK/user_data_tiktok.json"
PARQUET_DIR = "../../../data/parquet"
MY_USERNAME = "arnvudl"  # ← ton username TikTok

def anonymize(val, salt="mydigitaltwin"):
    return hashlib.sha256(f"{salt}{val}".encode()).hexdigest()[:10]

def parse_dt(date_str):
    """Parse une date TikTok 'YYYY-MM-DD HH:MM:SS' en timestamp ms."""
    if not date_str:
        return 0
    try:
        dt = datetime.strptime(date_str.strip(), "%Y-%m-%d %H:%M:%S")
        return int(dt.replace(tzinfo=timezone.utc).timestamp() * 1000)
    except:
        return 0

def extract_video_id(url):
    """Extrait l'ID vidéo depuis une URL TikTok."""
    import re
    m = re.search(r'video/(\d+)', url or '')
    return m.group(1) if m else ""

Spark version : 3.5.5


26/03/28 13:46:24 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# Chargement du JSON (~37 Mo)
print("Chargement du JSON TikTok...")
with open(RAW_PATH, encoding="utf-8", errors="replace") as f:
    data = json.load(f)

print("✓ Chargé — clés racine :", list(data.keys()))

Chargement du JSON TikTok...
✓ Chargé — clés racine : ['Comment', 'Direct Message', 'Likes and Favorites', 'Post', 'Profile And Settings', 'Your Activity']


---
## PARTIE 1 — Watch History
### 1.1 Ingestion

In [3]:
raw_watch = data.get("Your Activity", {}).get("Watch History", {}).get("VideoList", [])
print(f"Vidéos regardées : {len(raw_watch):,}")

watch_rows = []
for item in raw_watch:
    ts_ms    = parse_dt(item.get("Date", ""))
    url      = item.get("Link", "")
    video_id = extract_video_id(url)
    watch_rows.append({
        "video_id":          video_id,
        "url":               url,
        "timestamp_ms":      ts_ms,
        "interaction_weight": 1.0,
        "action_type":       "watch",
        "platform":          "tiktok",
    })

schema_watch = StructType([
    StructField("video_id",           StringType(), True),
    StructField("url",                StringType(), True),
    StructField("timestamp_ms",       LongType(),   True),
    StructField("interaction_weight", DoubleType(), True),
    StructField("action_type",        StringType(), True),
    StructField("platform",           StringType(), True),
])

df_watch = spark.createDataFrame(watch_rows, schema=schema_watch) \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date"))

print(f"DataFrame : {df_watch.count():,} lignes")
df_watch.show(5, truncate=60)

Vidéos regardées : 234,771


26/03/28 13:46:45 WARN TaskSetManager: Stage 0 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.


DataFrame : 234,771 lignes


26/03/28 13:46:52 WARN TaskSetManager: Stage 3 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.


+-------------------+--------------------------------------------------------+-------------+------------------+-----------+--------+-------------------+----------+-----------+----------+-------------+
|           video_id|                                                     url| timestamp_ms|interaction_weight|action_type|platform|         event_date|event_year|event_month|event_hour|event_weekday|
+-------------------+--------------------------------------------------------+-------------+------------------+-----------+--------+-------------------+----------+-----------+----------+-------------+
|7442006951443090721|https://www.tiktokv.com/share/video/7442006951443090721/|1759678617000|               1.0|      watch|  tiktok|2025-10-05 15:36:57|      2025|    2025-10|        15|            1|
|7283205346695433505|https://www.tiktokv.com/share/video/7283205346695433505/|1759678631000|               1.0|      watch|  tiktok|2025-10-05 15:37:11|      2025|    2025-10|        15|          

### 1.2 Exploration

In [4]:
print("=== Vidéos regardées par année ===")
df_watch.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Activité par heure ===")
df_watch.groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Mois les plus actifs ===")
df_watch.groupBy("event_month") \
    .count().orderBy(F.desc("count")).limit(10).show()

=== Vidéos regardées par année ===


26/03/28 13:47:07 WARN TaskSetManager: Stage 4 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.


+----------+------+
|event_year| count|
+----------+------+
|      2024|  3426|
|      2025|186000|
|      2026| 45345|
+----------+------+


=== Activité par heure ===


26/03/28 13:47:10 WARN TaskSetManager: Stage 7 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.


+----------+-----+
|event_hour|count|
+----------+-----+
|         0|11976|
|         1| 9072|
|         2| 4873|
|         3| 2028|
|         4|  389|
|         5| 2576|
|         6| 3027|
|         7| 4514|
|         8| 9151|
|         9|11550|
|        10|15618|
|        11|12833|
|        12|12407|
|        13| 8288|
|        14| 8610|
|        15|11250|
|        16|11446|
|        17|11060|
|        18|15505|
|        19|15491|
+----------+-----+
only showing top 20 rows


=== Mois les plus actifs ===


26/03/28 13:47:11 WARN TaskSetManager: Stage 10 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.


+-----------+-----+
|event_month|count|
+-----------+-----+
|    2025-12|22328|
|    2026-01|18242|
|    2026-02|17908|
|    2025-05|15709|
|    2025-08|15695|
|    2025-03|15664|
|    2025-04|15496|
|    2025-07|15345|
|    2025-11|15177|
|    2025-09|14819|
+-----------+-----+



### 1.3 Écriture Parquet

In [5]:
df_watch.write.mode("overwrite").parquet(f"{PARQUET_DIR}/tiktok_watch.parquet")
print(f"✓ tiktok_watch.parquet — {df_watch.count():,} lignes")

26/03/28 13:47:27 WARN TaskSetManager: Stage 13 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.
26/03/28 13:47:36 WARN TaskSetManager: Stage 14 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.


✓ tiktok_watch.parquet — 234,771 lignes


---
## PARTIE 2 — Likes
### 2.1 Ingestion

In [6]:
raw_likes = data.get("Likes and Favorites", {}).get("Like List", {}).get("ItemFavoriteList", [])
print(f"Likes : {len(raw_likes):,}")

like_rows = []
for item in raw_likes:
    ts_ms    = parse_dt(item.get("date", ""))
    url      = item.get("link", "")
    video_id = extract_video_id(url)
    like_rows.append({
        "video_id":           video_id,
        "url":                url,
        "timestamp_ms":       ts_ms,
        "interaction_weight": 2.0,
        "action_type":        "like",
        "platform":           "tiktok",
    })

df_likes = spark.createDataFrame(like_rows, schema=schema_watch) \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date"))

print(f"DataFrame : {df_likes.count():,} lignes")
df_likes.show(5, truncate=60)

Likes : 6,000
DataFrame : 6,000 lignes
+-------------------+--------------------------------------------------------+-------------+------------------+-----------+--------+-------------------+----------+-----------+----------+-------------+
|           video_id|                                                     url| timestamp_ms|interaction_weight|action_type|platform|         event_date|event_year|event_month|event_hour|event_weekday|
+-------------------+--------------------------------------------------------+-------------+------------------+-----------+--------+-------------------+----------+-----------+----------+-------------+
|7608904131767438614|https://www.tiktokv.com/share/video/7608904131767438614/|1774008169000|               2.0|       like|  tiktok|2026-03-20 12:02:49|      2026|    2026-03|        12|            6|
|7619013106454301974|https://www.tiktokv.com/share/video/7619013106454301974/|1774008139000|               2.0|       like|  tiktok|2026-03-20 12:02:19|     

### 2.2 Exploration

In [7]:
print("=== Likes par année ===")
df_likes.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Activité par heure ===")
df_likes.groupBy("event_hour").count().orderBy("event_hour").show()

=== Likes par année ===


+----------+-----+
|event_year|count|
+----------+-----+
|      2026| 6000|
+----------+-----+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|  348|
|         1|  282|
|         2|  186|
|         3|  138|
|         4|    6|
|         5|   98|
|         6|   30|
|         7|   91|
|         8|   80|
|         9|  250|
|        10|  237|
|        11|  361|
|        12|  316|
|        13|  206|
|        14|  167|
|        15|  231|
|        16|  209|
|        17|  190|
|        18|  457|
|        19|  561|
+----------+-----+
only showing top 20 rows



### 2.3 Écriture Parquet

In [8]:
df_likes.write.mode("overwrite").parquet(f"{PARQUET_DIR}/tiktok_likes.parquet")
print(f"✓ tiktok_likes.parquet — {df_likes.count():,} lignes")

✓ tiktok_likes.parquet — 6,000 lignes


---
## PARTIE 3 — Recherches
### 3.1 Ingestion

In [9]:
raw_searches = data.get("Your Activity", {}).get("Searches", {}).get("SearchList", [])
print(f"Recherches : {len(raw_searches):,}")

search_rows = []
for item in raw_searches:
    ts_ms = parse_dt(item.get("Date", ""))
    query = item.get("SearchTerm", "")
    search_rows.append({
        "query":        query,
        "timestamp_ms": ts_ms,
        "char_count":   len(query),
        "word_count":   len(query.split()),
        "platform":     "tiktok",
    })

schema_search = StructType([
    StructField("query",        StringType(), True),
    StructField("timestamp_ms", LongType(),   True),
    StructField("char_count",   IntegerType(), True),
    StructField("word_count",   IntegerType(), True),
    StructField("platform",     StringType(), True),
])

df_searches = spark.createDataFrame(search_rows, schema=schema_search) \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date"))

print(f"DataFrame : {df_searches.count():,} lignes")
df_searches.show(10, truncate=50)

Recherches : 1,053
DataFrame : 1,053 lignes
+-----------------------------+-------------+----------+----------+--------+-------------------+----------+-----------+----------+-------------+
|                        query| timestamp_ms|char_count|word_count|platform|         event_date|event_year|event_month|event_hour|event_weekday|
+-----------------------------+-------------+----------+----------+--------+-------------------+----------+-----------+----------+-------------+
|              anyme bb jacque|1759678546000|        15|         3|  tiktok|2025-10-05 15:35:46|      2025|    2025-10|        15|            1|
|                     wearline|1759678554000|         8|         1|  tiktok|2025-10-05 15:35:54|      2025|    2025-10|        15|            1|
|              anyme bb jacque|1761745215000|        15|         3|  tiktok|2025-10-29 13:40:15|      2025|    2025-10|        13|            4|
|          free excel tempkate|1761745229000|        19|         3|  tiktok|2025-10-29

### 3.2 Exploration

In [10]:
print("=== Top 20 termes recherchés ===")
df_searches.groupBy("query") \
    .count().orderBy(F.desc("count")).limit(20).show(truncate=50)

print("\n=== Top 20 mots recherchés ===")
df_searches \
    .withColumn("word", F.explode(F.split(F.lower("query"), r"\s+"))) \
    .filter(F.length("word") > 1) \
    .groupBy("word").count() \
    .orderBy(F.desc("count")).limit(20).show()

print("\n=== Recherches par année ===")
df_searches.groupBy("event_year").count().orderBy("event_year").show()

=== Top 20 termes recherchés ===


+------------------------------+-----+
|                         query|count|
+------------------------------+-----+
|                           PGE|    5|
|                      anyme023|    5|
|                  playboicarti|    5|
|               anyme bb jacque|    4|
|freed from desire x woops edit|    4|
|                       Maxence|    4|
|                    Malien🇫🇷|    4|
|                      hattrick|    3|
|                    92 et puis|    3|
|                        flamby|    3|
|   Jsuis dqns le bat a djibril|    3|
|                       Et puis|    3|
|              chant alcoolique|    3|
|                Pack sample dj|    3|
|    Cosplay vladimir cauchemar|    3|
|     tas a tocar bue mas mesmo|    3|
|           big mama transition|    3|
|           Remove water iphone|    2|
|                    aw dang it|    2|
|              evil jordan edit|    2|
+------------------------------+-----+


=== Top 20 mots recherchés ===


+----------+-----+
|      word|count|
+----------+-----+
|        la|   40|
|        dj|   35|
|       pas|   26|
|      edit|   25|
|     anyme|   24|
|     dance|   23|
|transition|   22|
|        to|   20|
|        on|   19|
|        le|   17|
|        je|   17|
|       the|   16|
|        de|   16|
|      song|   15|
|  tutorial|   14|
|      from|   12|
|  original|   12|
|       ref|   12|
|    serato|   12|
|        du|   11|
+----------+-----+


=== Recherches par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2024|   20|
|      2025|  816|
|      2026|  217|
+----------+-----+



### 3.3 Écriture Parquet

In [11]:
df_searches.write.mode("overwrite").parquet(f"{PARQUET_DIR}/tiktok_searches.parquet")
print(f"✓ tiktok_searches.parquet — {df_searches.count():,} lignes")

✓ tiktok_searches.parquet — 1,053 lignes


---
## PARTIE 4 — Commentaires
### 4.1 Ingestion

In [12]:
raw_comments = data.get("Comment", {}).get("Comments", {}).get("CommentsList", [])
print(f"Commentaires : {len(raw_comments):,}")

comment_rows = []
for item in raw_comments:
    ts_ms   = parse_dt(item.get("date", ""))
    comment = item.get("comment", "")
    url     = item.get("url", "")
    comment_rows.append({
        "text":         comment,
        "url":          url,
        "timestamp_ms": ts_ms,
        "char_count":   len(comment),
        "word_count":   len(comment.split()),
        "platform":     "tiktok",
        "content_type": "comment",
    })

schema_comments = StructType([
    StructField("text",         StringType(),  True),
    StructField("url",          StringType(),  True),
    StructField("timestamp_ms", LongType(),    True),
    StructField("char_count",   IntegerType(), True),
    StructField("word_count",   IntegerType(), True),
    StructField("platform",     StringType(),  True),
    StructField("content_type", StringType(),  True),
])

df_comments = spark.createDataFrame(comment_rows, schema=schema_comments) \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date"))

print(f"DataFrame : {df_comments.count():,} lignes")
df_comments.show(5, truncate=60)

Commentaires : 635
DataFrame : 635 lignes
+------------------------------------------------------------+---+-------------+----------+----------+--------+------------+-------------------+----------+-----------+----------+-------------+
|                                                        text|url| timestamp_ms|char_count|word_count|platform|content_type|         event_date|event_year|event_month|event_hour|event_weekday|
+------------------------------------------------------------+---+-------------+----------+----------+--------+------------+-------------------+----------+-----------+----------+-------------+
|          C quoi ça ?? Comment ça il est mondial le p'tit ??|   |1772115558000|        50|        12|  tiktok|     comment|2026-02-26 14:19:18|      2026|    2026-02|        14|            5|
|C'est pas obligatoirement une image, NFT c'est pour jeton...|   |1770433090000|       120|        20|  tiktok|     comment|2026-02-07 02:58:10|      2026|    2026-02|         2|        

### 4.2 Exploration

In [13]:
print("=== Stats texte ===")
df_comments.agg(
    F.avg("char_count").alias("avg_chars"),
    F.avg("word_count").alias("avg_words"),
    F.max("char_count").alias("max_chars"),
).show()

print("\n=== Top 20 mots ===")
df_comments \
    .withColumn("word", F.explode(F.split(F.lower("text"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .filter(~F.col("word").startswith("http")) \
    .groupBy("word").count() \
    .orderBy(F.desc("count")).limit(20).show()

print("\n=== Commentaires par année ===")
df_comments.groupBy("event_year").count().orderBy("event_year").show()

=== Stats texte ===
+-----------------+-----------------+---------+
|        avg_chars|        avg_words|max_chars|
+-----------------+-----------------+---------+
|33.69133858267717|6.823622047244094|      150|
+-----------------+-----------------+---------+


=== Top 20 mots ===


+-----+-----+
| word|count|
+-----+-----+
|  pas|   89|
|  que|   53|
| mais|   45|
|  des|   43|
|  les|   42|
|c'est|   39|
|  est|   36|
| pour|   34|
| fait|   27|
|  qui|   26|
| j'ai|   25|
| dans|   24|
|  une|   24|
|  sur|   24|
|  dit|   23|
|  moi|   20|
|  son|   19|
|faire|   19|
|aussi|   18|
|  non|   17|
+-----+-----+


=== Commentaires par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2019|    1|
|      2020|  135|
|      2021|  397|
|      2022|   66|
|      2023|    6|
|      2024|    2|
|      2025|   24|
|      2026|    4|
+----------+-----+



### 4.3 Écriture Parquet

In [14]:
df_comments.write.mode("overwrite").parquet(f"{PARQUET_DIR}/tiktok_comments.parquet")
print(f"✓ tiktok_comments.parquet — {df_comments.count():,} lignes")

✓ tiktok_comments.parquet — 635 lignes


---
## PARTIE 5 — Messages (métadonnées + texte)

> **Deux Parquets** :
> - `tiktok_messages_meta.parquet` — anonymisé, pour K-Means temporel
> - `tiktok_messages_text.parquet` — tes messages uniquement, pour NLP Clone

### 5.1 Ingestion

In [15]:
chat_history = data.get("Direct Message", {}) \
                   .get("Direct Messages", {}) \
                   .get("ChatHistory", {})

print(f"Conversations : {len(chat_history):,}")

meta_rows = []
text_rows = []

for conv_key, messages in chat_history.items():
    # Extraire le nom de l'interlocuteur depuis la clé
    # ex: "Chat History with dj_yoyo206:"
    import re
    match = re.search(r'with\s+(.+?):', conv_key)
    other = match.group(1) if match else conv_key
    conv_id = anonymize(conv_key)

    if not isinstance(messages, list):
        continue

    for msg in messages:
        ts_ms   = parse_dt(msg.get("Date", ""))
        sender  = msg.get("From", "")
        content = msg.get("Content", "")
        is_me   = sender == MY_USERNAME

        # Détecter le type
        has_url = content.startswith("http") if content else False
        msg_type = "url" if has_url else ("text" if content else "other")

        # Métadonnées
        meta_rows.append({
            "conv_id":      conv_id,
            "sender_anon":  anonymize(sender),
            "is_me":        is_me,
            "timestamp_ms": ts_ms,
            "msg_type":     msg_type,
            "char_count":   len(content) if content else 0,
            "platform":     "tiktok",
        })

        # Texte — uniquement mes messages non-URL
        if is_me and content and not has_url:
            text_rows.append({
                "text":         content,
                "timestamp_ms": ts_ms,
                "platform":     "tiktok",
                "content_type": "dm",
            })

print(f"Messages (métadonnées) : {len(meta_rows):,}")
print(f"Mes messages (texte)   : {len(text_rows):,}")

Conversations : 12
Messages (métadonnées) : 11,869
Mes messages (texte)   : 4,337


In [16]:
# DataFrame métadonnées
schema_meta = StructType([
    StructField("conv_id",      StringType(),  True),
    StructField("sender_anon",  StringType(),  True),
    StructField("is_me",        BooleanType(), True),
    StructField("timestamp_ms", LongType(),    True),
    StructField("msg_type",     StringType(),  True),
    StructField("char_count",   IntegerType(), True),
    StructField("platform",     StringType(),  True),
])

df_meta = spark.createDataFrame(meta_rows, schema=schema_meta) \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date"))

# DataFrame texte
schema_text = StructType([
    StructField("text",         StringType(), True),
    StructField("timestamp_ms", LongType(),   True),
    StructField("platform",     StringType(), True),
    StructField("content_type", StringType(), True),
])

df_text = spark.createDataFrame(text_rows, schema=schema_text) \
    .withColumn("event_date",  F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("char_count",  F.length("text")) \
    .withColumn("word_count",  F.size(F.split(F.trim("text"), r"\s+")))

print(f"Métadonnées : {df_meta.count():,} lignes")
print(f"Texte       : {df_text.count():,} lignes")
df_meta.show(5)
df_text.show(5, truncate=60)

Métadonnées : 11,869 lignes
Texte       : 4,337 lignes
+----------+-----------+-----+-------------+--------+----------+--------+-------------------+----------+-----------+----------+-------------+
|   conv_id|sender_anon|is_me| timestamp_ms|msg_type|char_count|platform|         event_date|event_year|event_month|event_hour|event_weekday|
+----------+-----------+-----+-------------+--------+----------+--------+-------------------+----------+-----------+----------+-------------+
|433529bbab| 602455d85e| true|1773947747000|    text|        45|  tiktok|2026-03-19 19:15:47|      2026|    2026-03|        19|            5|
|433529bbab| 602455d85e| true|1773947747000|     url|        56|  tiktok|2026-03-19 19:15:47|      2026|    2026-03|        19|            5|
|433529bbab| 020053daa3|false|1773767860000|     url|        56|  tiktok|2026-03-17 17:17:40|      2026|    2026-03|        17|            3|
|433529bbab| 602455d85e| true|1772907565000|    text|        42|  tiktok|2026-03-07 18:19:25|

### 5.2 Exploration

In [17]:
print("=== Répartition par type de message ===")
df_meta.groupBy("msg_type").count().orderBy(F.desc("count")).show()

print("\n=== Moi vs les autres ===")
df_meta.groupBy("is_me").count().show()

print("\n=== Activité par heure ===")
df_meta.groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Top conversations (nb messages) ===")
df_meta.groupBy("conv_id") \
    .count().orderBy(F.desc("count")).limit(10).show()

print("\n=== Top 20 mots (mes messages) ===")
df_text \
    .withColumn("word", F.explode(F.split(F.lower("text"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .filter(~F.col("word").startswith("http")) \
    .groupBy("word").count() \
    .orderBy(F.desc("count")).limit(20).show()

=== Répartition par type de message ===
+--------+-----+
|msg_type|count|
+--------+-----+
|    text| 8638|
|     url| 3231|
+--------+-----+


=== Moi vs les autres ===
+-----+-----+
|is_me|count|
+-----+-----+
| true| 6108|
|false| 5761|
+-----+-----+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|  389|
|         1|  451|
|         2|  101|
|         3|   17|
|         4|   11|
|         5|   57|
|         6|   94|
|         7|  249|
|         8|  483|
|         9|  651|
|        10|  868|
|        11|  590|
|        12|  615|
|        13|  439|
|        14|  361|
|        15|  554|
|        16|  456|
|        17|  473|
|        18|  821|
|        19|  774|
+----------+-----+
only showing top 20 rows


=== Top conversations (nb messages) ===
+----------+-----+
|   conv_id|count|
+----------+-----+
|c45362e49b| 5930|
|7cc9af6c53| 2344|
|3e11724c44| 1443|
|8b0a991826|  899|
|3c7b1bbf91|  597|
|3801e80092|  397|
|433529bbab|  141|
|dfa

+-----+-----+
| word|count|
+-----+-----+
|  pas|  540|
| mais|  346|
|  que|  283|
|  les|  267|
| j'ai|  263|
|  toi|  232|
|c'est|  231|
|  moi|  165|
|  une|  163|
| pour|  149|
|  des|  145|
|  qui|  138|
|  est|  133|
| plus|  131|
|  bah|  130|
| fait|  118|
| dans|  118|
|comme|  114|
|  nan|  105|
|faire|  104|
+-----+-----+



### 5.3 Écriture Parquet

In [18]:
df_meta.write.mode("overwrite").parquet(f"{PARQUET_DIR}/tiktok_messages_meta.parquet")
print(f"✓ tiktok_messages_meta.parquet — {df_meta.count():,} lignes")

df_text.write.mode("overwrite").parquet(f"{PARQUET_DIR}/tiktok_messages_text.parquet")
print(f"✓ tiktok_messages_text.parquet — {df_text.count():,} lignes")

✓ tiktok_messages_meta.parquet — 11,869 lignes


✓ tiktok_messages_text.parquet — 4,337 lignes


---
## Résumé

In [19]:
print("=" * 58)
print("  MyDigitalTwin — TikTok — Résumé")
print("=" * 58)
print(f"  Watch history  : {df_watch.count():>8,} → tiktok_watch.parquet")
print(f"  Likes          : {df_likes.count():>8,} → tiktok_likes.parquet")
print(f"  Recherches     : {df_searches.count():>8,} → tiktok_searches.parquet")
print(f"  Commentaires   : {df_comments.count():>8,} → tiktok_comments.parquet")
print(f"  Messages meta  : {df_meta.count():>8,} → tiktok_messages_meta.parquet")
print(f"  Messages texte : {df_text.count():>8,} → tiktok_messages_text.parquet")
print("=" * 58)
print()
print("  Usage ML :")
print("  - Watch + Likes    → ALS (axe 2, weight=1/2)")
print("  - Commentaires     → NLP Clone (axe 1)")
print("  - Messages texte   → NLP Clone (axe 1)")
print("  - Messages meta    → K-Means temporel (axe 3)")

  MyDigitalTwin — TikTok — Résumé


26/03/28 13:50:33 WARN TaskSetManager: Stage 96 contains a task of very large size (1059 KiB). The maximum recommended task size is 1000 KiB.


  Watch history  :  234,771 → tiktok_watch.parquet
  Likes          :    6,000 → tiktok_likes.parquet
  Recherches     :    1,053 → tiktok_searches.parquet
  Commentaires   :      635 → tiktok_comments.parquet
  Messages meta  :   11,869 → tiktok_messages_meta.parquet
  Messages texte :    4,337 → tiktok_messages_text.parquet

  Usage ML :
  - Watch + Likes    → ALS (axe 2, weight=1/2)
  - Commentaires     → NLP Clone (axe 1)
  - Messages texte   → NLP Clone (axe 1)
  - Messages meta    → K-Means temporel (axe 3)


In [20]:
spark.stop()